## Build Your First AI Agent (No Strands)

This notebook shows how to build a simple tool-using “agent” **without** the `strands` / `strands_tools` framework.

- **LLM model**: Amazon Bedrock via `boto3` (`bedrock-runtime`)
- **Agentic loop**: a tiny ReAct-style loop (model decides an `Action`, we run a Python tool, then feed back an `Observation`)
- **Tools**: `current_time`, `http_request`, `calculator`, `letter_counter`


![Image](https://d2908q01vomqb2.cloudfront.net/ca3512f4dfa95a03169c5a670a4c91a19b3077b4/2025/05/16/agentic-loop.png)

## Install libraries

You only need `boto3` to talk to Bedrock, plus `requests` for the HTTP tool.


In [1]:
%pip install -q boto3 requests


Note: you may need to restart the kernel to use updated packages.


## Configure AWS credentials 



Set these environment variables before running, or configure an AWS profile:

- `AWS_ACCESS_KEY_ID`
- `AWS_SECRET_ACCESS_KEY`
- `AWS_REGION` (e.g. `ap-southeast-1`)

Then pick a Bedrock `model_id` that exists in your region (we’ll list them next).


In [ ]:
# copy the info here from the aws_credential.txt file

In [ ]:
## set up aws access key and secret key
import os
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_REGION"] = AWS_REGION

## set up aws profile
os.environ["AWS_PROFILE"] = "default"

In [2]:
import os

AWS_REGION = os.getenv("AWS_REGION", "ap-southeast-1")

# Choose a default model_id that commonly exists (verify using the next cell!)
MODEL_ID = os.getenv("AWS_BEDROCK_MODEL_ID", "apac.anthropic.claude-sonnet-4-20250514-v1:0")

print("AWS_REGION:", AWS_REGION)
print("MODEL_ID:", MODEL_ID)
print("Has AWS_ACCESS_KEY_ID:", bool(os.getenv("AWS_ACCESS_KEY_ID")))
print("Has AWS_PROFILE:", bool(os.getenv("AWS_PROFILE")))



AWS_REGION: ap-southeast-1
MODEL_ID: apac.anthropic.claude-sonnet-4-20250514-v1:0
Has AWS_ACCESS_KEY_ID: False
Has AWS_PROFILE: False


## LLM models

In [3]:
import boto3

bedrock = boto3.client("bedrock", region_name=AWS_REGION)
resp = bedrock.list_foundation_models()
models = resp.get("modelSummaries", [])

print(f"Found {len(models)} foundation models in {AWS_REGION}\n")

for m in sorted(models, key=lambda x: x.get("modelId", "")):
    print(m.get("modelId"))

print("\n--- filtered: TEXT output models ---")
for m in sorted(models, key=lambda x: x.get("modelId", "")):
    if "TEXT" in (m.get("outputModalities") or []):
        print(m.get("modelId"))



Found 19 foundation models in ap-southeast-1

amazon.nova-2-lite-v1:0
amazon.nova-lite-v1:0
amazon.nova-micro-v1:0
amazon.nova-pro-v1:0
anthropic.claude-3-5-sonnet-20240620-v1:0
anthropic.claude-3-5-sonnet-20241022-v2:0
anthropic.claude-3-7-sonnet-20250219-v1:0
anthropic.claude-3-haiku-20240307-v1:0
anthropic.claude-3-sonnet-20240229-v1:0
anthropic.claude-3-sonnet-20240229-v1:0:200k
anthropic.claude-3-sonnet-20240229-v1:0:28k
anthropic.claude-haiku-4-5-20251001-v1:0
anthropic.claude-opus-4-5-20251101-v1:0
anthropic.claude-sonnet-4-20250514-v1:0
anthropic.claude-sonnet-4-5-20250929-v1:0
cohere.embed-english-v3
cohere.embed-multilingual-v3
cohere.embed-v4:0
twelvelabs.pegasus-1-2-v1:0

--- filtered: TEXT output models ---
amazon.nova-2-lite-v1:0
amazon.nova-lite-v1:0
amazon.nova-micro-v1:0
amazon.nova-pro-v1:0
anthropic.claude-3-5-sonnet-20240620-v1:0
anthropic.claude-3-5-sonnet-20241022-v2:0
anthropic.claude-3-7-sonnet-20250219-v1:0
anthropic.claude-3-haiku-20240307-v1:0
anthropic.claud

## Tools

We’ll implement a few tiny Python tools, then let the model decide when to use them.


In [4]:
from __future__ import annotations

import ast
import datetime as _dt
import math
from typing import Any, Callable

import requests


def current_time() -> str:
    """Return current UTC time in ISO-8601."""
    return _dt.datetime.now(tz=_dt.timezone.utc).isoformat()


def http_request(url: str, method: str = "GET", headers: dict[str, str] | None = None, timeout: int = 20) -> str:
    """Fetch a URL and return a short text snippet (status + first chars)."""
    method = method.upper().strip()
    headers = headers or {}

    # Many sites (e.g. Wikipedia) block empty UA.
    headers.setdefault("User-Agent", "IT3103 Teaching Bot/1.0")

    resp = requests.request(method=method, url=url, headers=headers, timeout=timeout)
    snippet = resp.text[:1500]
    return f"status={resp.status_code}\n\n{snippet}"


_ALLOWED_MATH_FUNCS: dict[str, Callable[..., Any]] = {
    "abs": abs,
    "round": round,
    "min": min,
    "max": max,
    "sum": sum,
    "pow": pow,
}

_ALLOWED_MATH_ATTRS = {
    "sqrt",
    "log",
    "log10",
    "exp",
    "sin",
    "cos",
    "tan",
    "pi",
    "e",
}


def _safe_eval_expr(node: ast.AST) -> Any:
    if isinstance(node, ast.Expression):
        return _safe_eval_expr(node.body)

    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("only int/float constants are allowed")

    if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
        v = _safe_eval_expr(node.operand)
        return +v if isinstance(node.op, ast.UAdd) else -v

    if isinstance(node, ast.BinOp) and isinstance(
        node.op, (ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv, ast.Mod, ast.Pow)
    ):
        left = _safe_eval_expr(node.left)
        right = _safe_eval_expr(node.right)
        return {
            ast.Add: lambda a, b: a + b,
            ast.Sub: lambda a, b: a - b,
            ast.Mult: lambda a, b: a * b,
            ast.Div: lambda a, b: a / b,
            ast.FloorDiv: lambda a, b: a // b,
            ast.Mod: lambda a, b: a % b,
            ast.Pow: lambda a, b: a**b,
        }[type(node.op)](left, right)

    if isinstance(node, ast.Tuple):
        return tuple(_safe_eval_expr(elt) for elt in node.elts)

    if isinstance(node, ast.Name):
        if node.id == "math":
            return math
        raise ValueError(f"name not allowed: {node.id}")

    if isinstance(node, ast.Attribute):
        base = _safe_eval_expr(node.value)
        if base is math and node.attr in _ALLOWED_MATH_ATTRS:
            return getattr(math, node.attr)
        raise ValueError("attribute not allowed")

    if isinstance(node, ast.Call):
        if node.keywords:
            raise ValueError("keyword arguments not allowed")

        func = _safe_eval_expr(node.func)
        args = [_safe_eval_expr(a) for a in node.args]

        # allow calling only whitelisted builtins or math.<attr>
        if func in _ALLOWED_MATH_FUNCS.values() or (getattr(func, "__module__", None) == "math"):
            return func(*args)
        raise ValueError("function not allowed")

    raise ValueError(f"unsupported expression node: {type(node).__name__}")


def calculator(expression: str) -> str:
    """Safely evaluate simple arithmetic expressions (no variables, no imports)."""
    expression = expression.strip()
    if not expression:
        raise ValueError("empty expression")

    tree = ast.parse(expression, mode="eval")
    return str(_safe_eval_expr(tree))


def letter_counter(word: str, letter: str) -> int:
    """Count occurrences of a specific letter in a word."""
    if not isinstance(word, str) or not isinstance(letter, str):
        return 0
    if len(letter) != 1:
        raise ValueError("'letter' must be a single character")
    return word.lower().count(letter.lower())


TOOLS: dict[str, Callable[..., Any]] = {
    "current_time": current_time,
    "http_request": http_request,
    "calculator": calculator,
    "letter_counter": letter_counter,
}

print("Available tools:", sorted(TOOLS.keys()))



Available tools: ['calculator', 'current_time', 'http_request', 'letter_counter']


## Bedrock chat call (no framework)

We will call Bedrock using `boto3.client("bedrock-runtime")` and the `converse()` API.


In [5]:
import boto3

bedrock_rt = boto3.client("bedrock-runtime", region_name=AWS_REGION)


def bedrock_chat(system_prompt: str, messages: list[dict[str, Any]], *, temperature: float = 0.0, max_tokens: int = 800) -> str:
    """Send messages to Bedrock (Converse) and return assistant text."""
    resp = bedrock_rt.converse(
        modelId=MODEL_ID,
        system=[{"text": system_prompt}],
        messages=messages,
        inferenceConfig={"temperature": temperature, "maxTokens": max_tokens},
    )

    out = resp.get("output", {}).get("message", {}).get("content", [])
    # concatenate all returned text parts
    return "".join(part.get("text", "") for part in out)



## Agentic loop (ReAct-style)

The model must respond in **one** of these formats:

**Tool use**:

```
Thought: ...
Action: tool_name
Action Input: {"arg1": "..."}
```

**Final answer**:

```
Final: ...
```

We parse `Action` / `Action Input`, run the tool in Python, then feed back an `Observation`.


In [6]:
import json
import re
from typing import Callable, Optional


# callback(event, payload)
Callback = Callable[[str, dict[str, Any]], None]


SYSTEM_PROMPT = """You are a helpful assistant.

You can use tools when needed.

TOOLS AVAILABLE:
{tool_list}

When you decide to use a tool, you MUST output exactly (and NOTHING else):
Thought: <short>
Action: <tool_name>
Action Input: <valid JSON object>

Rules:
- Call **at most one tool per message**.
- Do NOT include `Final:` in the same message as an `Action:`.
- Do NOT include `Observation:` in your message. The system will provide the Observation after running the tool.

When you are ready to answer the user, output exactly:
Final: <your answer>
"""


_ACTION_RE = re.compile(r"^Action:\s*(?P<name>\w+)\s*$", re.MULTILINE)
_FINAL_RE = re.compile(r"^Final:\s*(?P<final>[\s\S]+)$", re.MULTILINE)


def _tool_list_text() -> str:
    lines = []
    for name, fn in sorted(TOOLS.items()):
        doc = (fn.__doc__ or "").strip()
        doc = doc.splitlines()[0] if doc else ""
        lines.append(f"- {name}: {doc}")
    return "\n".join(lines)


def _extract_first_json_object(s: str) -> str:
    """Extract the first JSON object from a string using balanced braces."""
    start = s.find("{")
    if start == -1:
        raise ValueError("no '{' found for JSON object")

    i = start
    depth = 0
    in_str = False
    esc = False

    while i < len(s):
        ch = s[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return s[start : i + 1]
        i += 1

    raise ValueError("unterminated JSON object")


def _parse_action(model_text: str) -> tuple[Optional[str], Optional[dict[str, Any]], Optional[str]]:
    """Return (tool_name, tool_kwargs, final_text).

    Notes:
    - If the model outputs BOTH tool-use and Final in the same message, we prioritize tool-use.
    - If the model outputs multiple tool calls, we take ONLY the first one.
    """
    m_action = _ACTION_RE.search(model_text)
    if m_action:
        tool_name = m_action.group("name")

        # Find the first Action Input after this Action
        after_action = model_text[m_action.end() :]
        m_ai = re.search(r"^Action Input:\s*(?P<rest>[\s\S]+)$", after_action, re.MULTILINE)
        if not m_ai:
            raise ValueError("Action Input not found after Action")

        json_text = _extract_first_json_object(m_ai.group("rest"))
        tool_kwargs = json.loads(json_text)
        if not isinstance(tool_kwargs, dict):
            raise ValueError("Action Input must be a JSON object")

        return tool_name, tool_kwargs, None

    m_final = _FINAL_RE.search(model_text)
    if m_final:
        return None, None, m_final.group("final").strip()

    return None, None, None


def run_agent(
    user_prompt: str,
    *,
    max_steps: int = 6,
    temperature: float = 0.0,
    callback: Callback | None = None,
) -> str:
    """Run a tiny tool-using loop. If callback is set, it will receive tool calls + observations."""
    tool_list = _tool_list_text()
    system_prompt = SYSTEM_PROMPT.format(tool_list=tool_list)

    messages: list[dict[str, Any]] = [
        {"role": "user", "content": [{"text": user_prompt}]},
    ]

    used_any_tool = False

    for step in range(1, max_steps + 1):
        model_text = bedrock_chat(system_prompt, messages, temperature=temperature)
        if callback:
            callback("model_output", {"step": step, "text": model_text})

        tool_name, tool_kwargs, final_text = _parse_action(model_text)
        if final_text is not None:
            if callback and not used_any_tool:
                callback("info", {"step": step, "info": "No tools were used (so no Observation was produced)."})
            if callback:
                callback("final", {"step": step, "final": final_text})
            return final_text

        if tool_name is None:
            # model didn't follow the contract; treat as final
            if callback and not used_any_tool:
                callback("info", {"step": step, "info": "No tools were used (so no Observation was produced)."})
            if callback:
                callback("final", {"step": step, "final": model_text.strip()})
            return model_text.strip()

        used_any_tool = True

        if callback:
            callback("tool_call", {"step": step, "tool": tool_name, "input": tool_kwargs})

        if tool_name not in TOOLS:
            observation = f"ERROR: unknown tool '{tool_name}'. Available: {sorted(TOOLS.keys())}"
        else:
            try:
                result = TOOLS[tool_name](**tool_kwargs)
                observation = str(result)
            except Exception as e:
                observation = f"ERROR running {tool_name}: {type(e).__name__}: {e}"

        if callback:
            callback("observation", {"step": step, "tool": tool_name, "observation": observation})

        # Append assistant + observation so model can continue.
        messages.append({"role": "assistant", "content": [{"text": model_text}]})
        messages.append({"role": "user", "content": [{"text": f"Observation: {observation}"}]})

    if callback:
        callback("error", {"step": max_steps, "error": f"exceeded max_steps={max_steps}"})

    return f"ERROR: exceeded max_steps={max_steps}"



In [7]:
def print_callback(event: str, payload: dict[str, Any]) -> None:
    """Simple callback handler to print the agent loop."""
    step = payload.get("step", "?")

    if event == "model_output":
        print(f"\n=== Step {step}: Thought ===")
        print(payload["text"].strip())
    elif event == "tool_call":
        print(f"\n=== Step {step}: Action ===")
        print(payload["tool"], payload.get("input", {}))
    elif event == "observation":
        print(f"\n=== Step {step}: Observation ({payload.get('tool')}) ===")
        print(payload["observation"])
    elif event == "info":
        print(f"\n=== Info (step {step}) ===")
        print(payload.get("info"))
    elif event == "final":
        print(f"\n=== Final Answer (step {step}) ===")
        print(payload["final"])
    elif event == "error":
        print(f"\n=== Error (step {step}) ===")
        print(payload.get("error"))



## Demo 1: basic Q&A (no tools needed)


In [8]:
print(run_agent("Explain the concept of recursion in programming.", callback=print_callback))



=== Step 1: Thought ===
Recursion is a fundamental programming concept where a function calls itself to solve a problem by breaking it down into smaller, similar subproblems.

## Key Components of Recursion

**1. Base Case(s):** The condition that stops the recursion. Without this, the function would call itself infinitely.

**2. Recursive Case:** The part where the function calls itself with modified parameters, moving toward the base case.

## How Recursion Works

Think of recursion like a stack of function calls. Each recursive call is placed on top of the previous one, and they resolve from the top down once the base case is reached.

## Simple Example: Factorial

```python
def factorial(n):
    # Base case
    if n <= 1:
        return 1
    # Recursive case
    else:
        return n * factorial(n - 1)
```

For `factorial(4)`:
- `factorial(4)` = 4 × `factorial(3)`
- `factorial(3)` = 3 × `factorial(2)`
- `factorial(2)` = 2 × `factorial(1)`
- `factorial(1)` = 1 (base case)
- Worki

## Demo 2: tool use (time + Wikipedia)




In [9]:
prompt = """
Answer the following questions:
1. What is the current time in UTC?
2. Based on Wikipedia, which CS concept can be traced back to Paul Bachmann?
""".strip()

print(run_agent(prompt, callback=print_callback))




=== Step 1: Thought ===
I'll help you answer both questions. Let me start by getting the current UTC time.

Thought: I need to get the current UTC time first.
Action: current_time
Action Input: {}

=== Step 1: Action ===
current_time {}

=== Step 1: Observation (current_time) ===
2026-01-20T06:08:08.595524+00:00

=== Step 2: Thought ===
Now let me search for information about Paul Bachmann on Wikipedia to answer the second question.

Thought: I need to fetch information from Wikipedia about Paul Bachmann to find out which CS concept can be traced back to him.
Action: http_request
Action Input: {"url": "https://en.wikipedia.org/wiki/Paul_Bachmann"}

=== Step 2: Action ===
http_request {'url': 'https://en.wikipedia.org/wiki/Paul_Bachmann'}

=== Step 2: Observation (http_request) ===
status=200

<!DOCTYPE html>
<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-to

## Demo 3: custom tools (calculator + letter_counter)


In [10]:
message = """
I have 4 requests:

1) What is the time right now?
2) Calculate 3111696 / 74088
3) Tell me how many letter R's are in the word "strawberry"
4) Then summarize all results in one short paragraph.
""".strip()

print(run_agent(message, callback=print_callback))




=== Step 1: Thought ===
I'll help you with all 4 requests. Let me start with getting the current time.

Thought: I need to get the current UTC time first.
Action: current_time
Action Input: {}

=== Step 1: Action ===
current_time {}

=== Step 1: Observation (current_time) ===
2026-01-20T06:08:16.762328+00:00

=== Step 2: Thought ===
Now let me calculate 3111696 / 74088.

Thought: I need to perform the division calculation.
Action: calculator
Action Input: {"expression": "3111696 / 74088"}

=== Step 2: Action ===
calculator {'expression': '3111696 / 74088'}

=== Step 2: Observation (calculator) ===
42.0

=== Step 3: Thought ===
Now let me count the letter R's in "strawberry".

Thought: I need to count how many times the letter 'r' appears in "strawberry".
Action: letter_counter
Action Input: {"word": "strawberry", "letter": "r"}

=== Step 3: Action ===
letter_counter {'word': 'strawberry', 'letter': 'r'}

=== Step 3: Observation (letter_counter) ===
3

=== Step 4: Thought ===
Final: He

In [11]:
message = """
I have 4 requests:

1) What is the time right now?
2) In 1988, a person's age was equal to the sum of the digits of their birth year. How old was this person?
3) Tell me how many letter R's are in the word "strawberry"
4) Then summarize all results in one short paragraph.
""".strip()

print(run_agent(message, callback=print_callback))


=== Step 1: Thought ===
I'll help you with all 4 requests. Let me start with the first one.

Thought: I need to get the current time.
Action: current_time
Action Input: {}

=== Step 1: Action ===
current_time {}

=== Step 1: Observation (current_time) ===
2026-01-20T06:08:25.902686+00:00

=== Step 2: Thought ===
Now let me work on the second request about the person's age in 1988.

Thought: I need to find a birth year where the sum of its digits equals the person's age in 1988. Let me calculate this step by step.
Action: calculator
Action Input: {"expression": "1988 - 1939"}

=== Step 2: Action ===
calculator {'expression': '1988 - 1939'}

=== Step 2: Observation (calculator) ===
49

=== Step 3: Thought ===
Let me verify this calculation by checking if the sum of digits in 1939 equals 49.

Thought: I need to calculate the sum of digits in 1939 to verify my answer.
Action: calculator
Action Input: {"expression": "1 + 9 + 3 + 9"}

=== Step 3: Action ===
calculator {'expression': '1 + 9 

## More examples

if you are interesting in learning coding an agent without using a framework (more freedom, more coding work), another good example can be found [here](https://github.com/leobeeson/single-file-ai-agent-tutorial/tree/master). The example is very similar to the one we use here.

